# **Modular AI Troubleshooting Agent**



In [8]:
import difflib  # used for similarity matching

# Knowledge Based

In [9]:
class KnowledgeBase:
    def __init__(self):
        """
        Stores known problems and their solutions.
        This acts as the 'memory' of the system.
        """
        self.issues = {
            "internet not working": "Restart router, check cables, and contact ISP.",
            "slow computer": "Close unused apps, clean disk, and scan for malware.",
            "printer not printing": "Check printer connection and reinstall drivers.",
            "wifi disconnecting": "Move closer to router or change WiFi channel."
        }

    def find_best_match(self, user_problem):
        """
        Finds the closest matching problem using similarity.
        This improves over exact keyword matching.
        """
        problems = list(self.issues.keys())

        # Get closest match using similarity
        match = difflib.get_close_matches(user_problem, problems, n=1, cutoff=0.5)

        if match:
            return match[0], self.issues[match[0]]
        return None, None

    def add_solution(self, problem, solution):
        """
        Learn new problem-solution pair.
        """
        self.issues[problem] = solution



# Trouble Shooting Agent

In [10]:
class TroubleshootingAgent:
    def __init__(self, knowledge_base):
        """
        Agent responsible for decision making.
        """
        self.kb = knowledge_base
        self.reward = 0  # performance score
        self.history = []  # store interactions

    def analyze_problem(self, problem):
        """
        Basic preprocessing (can upgrade to NLP later).
        """
        return problem.lower().strip()

    def decide_solution(self, problem):
        """
        Decision Engine:
        1. Try to find solution from knowledge base
        2. If not found → fallback response
        """
        matched_problem, solution = self.kb.find_best_match(problem)

        if solution:
            return solution, True, matched_problem
        else:
            return "No solution found. Escalating to human support.", False, None

    def update_reward(self, success):
        """
        Reward mechanism:
        +10 → success
        -5 → failure
        """
        if success:
            self.reward += 10
        else:
            self.reward -= 5

    def learn(self, problem, solution):
        """
        Learn from user feedback.
        """
        self.kb.add_solution(problem, solution)

# User Interface

In [11]:
class UserInterface:
    def get_input(self):
        return input("\nUser: ")

    def display(self, message):
        print("System:", message)

# Main System Loop

In [12]:
def main():
    print("=== Intelligent Troubleshooting Agent ===")

    # Initialize components
    kb = KnowledgeBase()
    agent = TroubleshootingAgent(kb)
    ui = UserInterface()

    # Run multiple interactions
    for _ in range(5):

        # Step 1: Get user problem
        problem = ui.get_input()

        # Step 2: Analyze input
        processed_problem = agent.analyze_problem(problem)

        # Step 3: Decide solution
        solution, success, matched = agent.decide_solution(processed_problem)

        # Step 4: Display solution
        ui.display(solution)

        # Step 5: Update reward
        agent.update_reward(success)

        # Step 6: Feedback loop (learning)
        feedback = input("Was this helpful? (yes/no): ")

        if feedback.lower() == "no":
            correct_solution = input("Please provide correct solution: ")
            agent.learn(processed_problem, correct_solution)
            print("✔ System learned a new solution!")

        # Store Interaction History
        agent.history.append((problem, solution, success))

    # Final Report
    print("\n=== SESSION SUMMARY ===")
    print("Total Reward:", agent.reward)

    print("\nInteraction History:")
    for record in agent.history:
        print(record)

In [14]:
if __name__ == "__main__":
    main()

=== Intelligent Troubleshooting Agent ===

User: internet issue
System: Restart router, check cables, and contact ISP.
Was this helpful? (yes/no): yes

User: slow pc
System: Close unused apps, clean disk, and scan for malware.
Was this helpful? (yes/no): yes

User: lagging
System: No solution found. Escalating to human support.
Was this helpful? (yes/no): yes

User: bugged computer
System: Close unused apps, clean disk, and scan for malware.
Was this helpful? (yes/no): no
Please provide correct solution: scan for viruses
✔ System learned a new solution!

User: 
System: No solution found. Escalating to human support.
Was this helpful? (yes/no): 

=== SESSION SUMMARY ===
Total Reward: 20

Interaction History:
('internet issue', 'Restart router, check cables, and contact ISP.', True)
('slow pc', 'Close unused apps, clean disk, and scan for malware.', True)
('lagging', 'No solution found. Escalating to human support.', False)
('bugged computer', 'Close unused apps, clean disk, and scan for